# Setup

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Determine device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define column containing the text
TEXT_COLUMN = 'prompt'

# Load datasets
df_train = pd.read_csv('/train.csv')

# Load tokenizers and models
deberta_name = "microsoft/deberta-v3-small"
roberta_name = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_name, num_labels=5).to(device)
deberta_model.eval()

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_name, num_labels=5).to(device)
roberta_model.eval()

# Label Mapping
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# Helper function to get probabilities for a single text
def get_probabilities(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return F.softmax(logits, dim=-1)[0]

# Pre-compute probabilities for row 25
row_25_text = df_train[TEXT_COLUMN].iloc[25]
deberta_probs = get_probabilities(row_25_text, deberta_model, deberta_tokenizer)
roberta_probs = get_probabilities(row_25_text, roberta_model, roberta_tokenizer)

print("Setup complete. Probabilities for row 25 extracted.")

Using device: cuda


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Setup complete. Probabilities for row 25 extracted.


# Question 1

In [2]:
# Extract the highest probability and corresponding label for DeBERTa
max_prob_deberta = torch.max(deberta_probs).item()
predicted_class_id_deberta = torch.argmax(deberta_probs).item()
predicted_option_deberta = label_map[predicted_class_id_deberta]

print(f"Highest Probability Option (DeBERTa): {predicted_option_deberta}")
print(f"Probability: {max_prob_deberta:.4f}")

Highest Probability Option (DeBERTa): A
Probability: 0.3027


# Question 2

In [3]:
# Average the class probabilities
avg_probs = (deberta_probs + roberta_probs) / 2

# Extract the highest averaged probability and corresponding label
max_avg_prob = torch.max(avg_probs).item()
predicted_class_id_avg = torch.argmax(avg_probs).item()
predicted_option_avg = label_map[predicted_class_id_avg]

print(f"Highest Averaged Probability Option: {predicted_option_avg}")
print(f"Probability: {max_avg_prob:.4f}")

Highest Averaged Probability Option: A
Probability: 0.2474


# Question 3

In [4]:
# Apply weighted averaging: 0.70 DeBERTa + 0.30 RoBERTa
weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)

# Extract the highest weighted probability and corresponding label
max_weighted_prob = torch.max(weighted_probs).item()
predicted_class_id_weighted = torch.argmax(weighted_probs).item()
predicted_option_weighted = label_map[predicted_class_id_weighted]

print(f"Highest Weighted Probability Option: {predicted_option_weighted}")
print(f"Probability: {max_weighted_prob:.4f}")

Highest Weighted Probability Option: A
Probability: 0.2695


# Question 4

In [5]:
# Rank the weighted probabilities to get the Top-3 indices
top_3_indices = torch.topk(weighted_probs, k=3).indices.tolist()

# Map the indices to labels (e.g., A, B, C) and join with spaces
top_3_options = [label_map[idx] for idx in top_3_indices]
top_3_string = " ".join(top_3_options)

print(f"Top-3 Prediction String for Row 25: {top_3_string}")

Top-3 Prediction String for Row 25: A D E


# Question 5

In [6]:
# Load test dataset
df_test = pd.read_csv('/test.csv')

predictions = []

# Iterate through test set
for idx, row in df_test.iterrows():
    text = row[TEXT_COLUMN]

    # Get probabilities
    deb_p = get_probabilities(text, deberta_model, deberta_tokenizer)
    rob_p = get_probabilities(text, roberta_model, roberta_tokenizer)

    # Weighted ensemble
    w_probs = (0.70 * deb_p) + (0.30 * rob_p)

    # Top-3 ranking
    top_3_idx = torch.topk(w_probs, k=3).indices.tolist()
    top_3_str = " ".join([label_map[i] for i in top_3_idx])

    # Store result (assuming test.csv has an 'id' column)
    predictions.append({'id': row['id'], 'prediction': top_3_str})

# Create submission file
submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission.csv', index=False)

# Count the number of prediction rows (excluding header)
num_prediction_rows = len(submission_df)

print(f"Submission saved to 'submission.csv'.")
print(f"Number of prediction rows: {num_prediction_rows}")

Submission saved to 'submission.csv'.
Number of prediction rows: 500


# Question 6

In [7]:
num_changed_tta = 0
tta_prefix = "Answer the following multiple-choice question carefully: "

for idx in range(min(50, len(df_train))):
    original_text = df_train[TEXT_COLUMN].iloc[idx]
    augmented_text = tta_prefix + original_text

    # Pass 1: Original prompt
    deb_p1 = get_probabilities(original_text, deberta_model, deberta_tokenizer)
    top1_original = torch.argmax(deb_p1).item()

    # Pass 2: Augmented prompt
    deb_p2 = get_probabilities(augmented_text, deberta_model, deberta_tokenizer)

    # Average probabilities
    avg_p_tta = (deb_p1 + deb_p2) / 2
    top1_tta = torch.argmax(avg_p_tta).item()

    # Compare Top-1 predictions
    if top1_original != top1_tta:
        num_changed_tta += 1

print(f"Number of rows with different Top-1 prediction after TTA: {num_changed_tta}")

Number of rows with different Top-1 prediction after TTA: 0


# Pre-compute probabilities for question 7-8

In [8]:
# Run inference once for the first 100 rows to save compute time
deberta_cached_probs = []
roberta_cached_probs = []
ensemble_cached_probs = []

limit = min(100, len(df_train))

for idx in range(limit):
    text = df_train[TEXT_COLUMN].iloc[idx]

    # Get probabilities
    deb_p = get_probabilities(text, deberta_model, deberta_tokenizer)
    rob_p = get_probabilities(text, roberta_model, roberta_tokenizer)

    # Weighted ensemble
    w_p = (0.70 * deb_p) + (0.30 * rob_p)

    # Cache results
    deberta_cached_probs.append(deb_p)
    roberta_cached_probs.append(rob_p)
    ensemble_cached_probs.append(w_p)

print(f"Cached probabilities for {limit} rows of train.csv.")

Cached probabilities for 100 rows of train.csv.


# Question 7

In [9]:
diff_top1_count = 0

for i in range(len(deberta_cached_probs)):
    deb_top1 = torch.argmax(deberta_cached_probs[i]).item()
    ens_top1 = torch.argmax(ensemble_cached_probs[i]).item()

    if deb_top1 != ens_top1:
        diff_top1_count += 1

print(f"Rows with different Top-1 predictions (DeBERTa vs Ensemble): {diff_top1_count}")

Rows with different Top-1 predictions (DeBERTa vs Ensemble): 0


# Question 8

In [10]:
positive_gain_count = 0

for i in range(len(deberta_cached_probs)):
    # Highest class probability (confidence)
    deb_conf = torch.max(deberta_cached_probs[i]).item()
    ens_conf = torch.max(ensemble_cached_probs[i]).item()

    confidence_gain = ens_conf - deb_conf

    if confidence_gain > 0:
        positive_gain_count += 1

print(f"Rows with positive confidence gain (> 0): {positive_gain_count}")

Rows with positive confidence gain (> 0): 0


# Question 9

In [11]:
ranking_change_count = 0

for i in range(len(deberta_cached_probs)):
    # Get Top-3 indices for both
    deb_top3 = torch.topk(deberta_cached_probs[i], k=3).indices.tolist()
    ens_top3 = torch.topk(ensemble_cached_probs[i], k=3).indices.tolist()

    # Compare the ordered lists
    if deb_top3 != ens_top3:
        ranking_change_count += 1

print(f"Rows with a change in Top-3 ranking: {ranking_change_count}")

Rows with a change in Top-3 ranking: 0


# Question 10

In [12]:
TRUE_LABEL_COLUMN = 'answer'

def calculate_map_at_3(predictions, true_label):
    """Calculates AP@3 for a single sample."""
    if true_label == predictions[0]:
        return 1.0
    elif true_label == predictions[1]:
        return 1.0 / 2.0
    elif true_label == predictions[2]:
        return 1.0 / 3.0
    else:
        return 0.0

total_score = 0.0
valid_rows_evaluated = 0

for i in range(len(ensemble_cached_probs)):
    # Try to fetch true label. If it doesn't exist, skip.
    try:
        true_label = df_train.iloc[i][TRUE_LABEL_COLUMN]
    except KeyError:
        print(f"Error: Column '{TRUE_LABEL_COLUMN}' not found in the dataset.")
        break

    # Get Top-3 options from the ensemble
    ens_top3_idx = torch.topk(ensemble_cached_probs[i], k=3).indices.tolist()
    ens_top3_options = [label_map[idx] for idx in ens_top3_idx]

    # Calculate score
    total_score += calculate_map_at_3(ens_top3_options, true_label)
    valid_rows_evaluated += 1

if valid_rows_evaluated > 0:
    map_at_3 = total_score / valid_rows_evaluated
    print(f"Final MAP@3 score: {map_at_3:.4f}")

Final MAP@3 score: 0.2933
